# Train v5c_full — Full Data Training (No Val Split)
**Based on v5c (drops + full constraints), trained on ALL training data.**

Best iterations from v5c (with val split):
- H1: 1036 → 1088 (+5%)
- H3: 423 → 444 (+5%)
- H10: 1300 → 1365 (+5%)
- H25: 780 → 819 (+5%)


In [1]:
import lightgbm as lgb
import pandas as pd
import numpy as np
import gc


In [2]:
import sys
sys.path.append('/home/lingod/kaggle_projects/ts_forecasting')
from src.score import weighted_rmse_score as score


In [3]:
def set_cat(df, cat_features):
    for feat in cat_features:
        df[feat] = df[feat].astype('category')
    return df


## Config

In [4]:
HORIZONS = [1, 3, 10, 25]

ZERO_CODES = ['MLAAMU3K', 'EP12UF2K', '1HEMHZK2', 'SJZP0OVU', '83EG83KQ']

VERSION = 'v5c_full'

# Best iterations from v5c + 5% buffer (rounded up)
BOOST_ROUNDS = {
    1:  1088,   # 1036 * 1.05 = 1087.8
    3:  444,    # 423 * 1.05 = 444.15
    10: 1365,   # 1300 * 1.05 = 1365.0
    25: 819,    # 780 * 1.05 = 819.0
}

params = {
    'objective': 'regression',
    'boosting_type': 'gbdt',
    'metric': 'rmse',
    'num_leaves': 80,
    'min_child_samples': 200,
    'lambda_l2': 10,
    'learning_rate': 0.01,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'feature_fraction': 0.8,
    'seed': 42,
    'verbosity': -1
}

# ── FEATURE DROPS (same as v5c) ──
GLOBAL_DROP = [
    'feature_aa', 'feature_au', 'feature_ax', 'feature_ba', 'feature_bb',
    'feature_bc', 'feature_bj', 'feature_bv', 'feature_c', 'feature_e',
    'feature_f'
]

PER_HORIZON_DROP = {
    1:  ['feature_ce', 'feature_x', 'feature_y'],
    3:  ['feature_ce','feature_y', 'feature_z'],
    10: ['feature_ce'],
    25: ['feature_ce'],
}

# ── FULL MONOTONE CONSTRAINTS (from IC analysis) ──
POSITIVE_CONSTRAINT = {
    1:  ['feature_cd', 'feature_ca'],
    3:  ['feature_cd', 'feature_ca'],
    10: ['feature_bz', 'feature_ca', 'feature_cb', 'feature_cc', 'feature_cd'],
    25: ['feature_al', 'feature_bz', 'feature_ca', 'feature_cb', 'feature_cc', 'feature_cd'],
}

NEGATIVE_CONSTRAINT = {
    1:  [],
    3:  ['feature_u', 'feature_bn'],
    10: ['feature_u', 'feature_v', 'feature_ag', 'feature_am',
         'feature_an', 'feature_ap', 'feature_bn'],
    25: ['feature_u', 'feature_v', 'feature_ag', 'feature_am',
         'feature_an', 'feature_ap', 'feature_bm', 'feature_bn', 'feature_bo'],
}


## Load & Prepare Data (Full — no split)

In [5]:
df = pd.read_parquet("/home/lingod/kaggle_projects/ts_forecasting/data/train.parquet")
cat_features = ['code', 'sub_category', 'horizon']

all_features = [feat for feat in df.columns
                if feat not in ['id', 'sub_code', 'ts_index', 'weight', 'y_target','horizon']]

base_features = [f for f in all_features if f not in GLOBAL_DROP]
print(f"Features: {len(all_features)} -> {len(base_features)} after global drops")
print(f"Training on FULL dataset: {len(df):,} rows (no val split)")

df = set_cat(df, cat_features)


Features: 88 -> 77 after global drops
Training on FULL dataset: 5,337,414 rows (no val split)


## Helper: Build Monotone Constraint Vector

In [6]:
def build_monotone_constraints(features, pos_list, neg_list):
    constraints = []
    for f in features:
        if f in pos_list:
            constraints.append(1)
        elif f in neg_list:
            constraints.append(-1)
        else:
            constraints.append(0)
    n_pos = sum(1 for c in constraints if c == 1)
    n_neg = sum(1 for c in constraints if c == -1)
    print(f"    Monotone constraints: {n_pos + n_neg} / {len(features)} features "
          f"(+1: {n_pos}, -1: {n_neg})")
    return constraints


## Train One Model per Horizon (Full Data, Fixed Rounds)

In [8]:
models      = {}
importances = {}
features_used = {}

for h in HORIZONS:
    print(f"\n{'='*60}")
    print(f"  Training horizon = {h}  |  num_boost_round = {BOOST_ROUNDS[h]}")
    print(f"{'='*60}")

    # Per-horizon feature list
    h_drops = PER_HORIZON_DROP.get(h, [])
    features = [f for f in base_features if f not in h_drops]
    features_used[h] = features
    print(f"  Features: {len(base_features)} -> {len(features)} after per-horizon drops")
    print(f"  Dropped for H={h}: {h_drops}")

    h_cat = [c for c in cat_features if c in features]

    # Build monotone constraints
    pos_list = POSITIVE_CONSTRAINT.get(h, [])
    neg_list = NEGATIVE_CONSTRAINT.get(h, [])
    mc = build_monotone_constraints(features, pos_list, neg_list)

    # Subset by horizon — FULL training data
    h_train = df[df['horizon'] == h].copy()
    print(f"  Train rows: {len(h_train):,} (full data)")

    # LightGBM Dataset — no validation set
    dtrain = lgb.Dataset(
        h_train[features],
        label=h_train['y_target'],
        weight=h_train['weight'],
        categorical_feature=h_cat,
        free_raw_data=False
    )

    # Add monotone constraints to params
    h_params = params.copy()
    if any(c != 0 for c in mc):
        h_params['monotone_constraints'] = mc

    # Train — fixed rounds, no early stopping
    model = lgb.train(
        h_params,
        dtrain,
        num_boost_round=BOOST_ROUNDS[h],
    )
    models[h] = model

    # Save model
    model_path = f"/home/lingod/kaggle_projects/ts_forecasting/models/lgb_model_{VERSION}_horizon{h}.txt"
    model.save_model(model_path)
    print(f"  Model saved -> {model_path}  (trained {BOOST_ROUNDS[h]} rounds)")

    # Feature importance
    imp = pd.DataFrame({
        'feature': features,
        'importance': model.feature_importance(importance_type='gain')
    }).sort_values('importance', ascending=False)
    importances[h] = imp
    print(f"\n  Top-10 features (horizon={h}):")
    print(imp.head(10).to_string(index=False))



  Training horizon = 1  |  num_boost_round = 1088
  Features: 77 -> 74 after per-horizon drops
  Dropped for H=1: ['feature_ce', 'feature_x', 'feature_y']
    Monotone constraints: 2 / 74 features (+1: 2, -1: 0)
  Train rows: 1,394,653 (full data)
  Model saved -> /home/lingod/kaggle_projects/ts_forecasting/models/lgb_model_v5c_full_horizon1.txt  (trained 1088 rounds)

  Top-10 features (horizon=1):
     feature   importance
  feature_al 3.990560e+06
sub_category 2.781255e+06
   feature_s 2.626323e+06
  feature_by 2.120253e+06
   feature_t 1.812905e+06
  feature_cf 1.810050e+06
   feature_z 1.734755e+06
   feature_v 1.636037e+06
   feature_w 1.571007e+06
  feature_cg 1.447345e+06

  Training horizon = 3  |  num_boost_round = 444
  Features: 77 -> 74 after per-horizon drops
  Dropped for H=3: ['feature_ce', 'feature_y', 'feature_z']
    Monotone constraints: 4 / 74 features (+1: 2, -1: 2)
  Train rows: 1,385,816 (full data)
  Model saved -> /home/lingod/kaggle_projects/ts_forecasting/m

## Train Scoring (sanity check on full training data)

In [ ]:
train_preds_series = pd.Series(index=df.index, dtype=float)

for h in HORIZONS:
    h_train = df[df['horizon'] == h].copy()
    model   = models[h]
    features = features_used[h]

    preds = model.predict(h_train[features])
    preds = pd.Series(preds, index=h_train.index)

    zero_mask = h_train['code'].isin(ZERO_CODES)
    preds[zero_mask] = 0.0
    train_preds_series[h_train.index] = preds

    h_score = score(h_train['y_target'], preds, h_train['weight'])
    print(f"  Horizon {h:>2d} | Train Score: {h_score:.4f}  (zero-coded rows: {zero_mask.sum():,})")

overall_train_score = score(df['y_target'], train_preds_series, df['weight'])
print(f"\n  Overall Train Score: {overall_train_score:.4f}")


In [9]:
del df
gc.collect()


1767

## Test Predictions

In [10]:
df_test = pd.read_parquet("/home/lingod/kaggle_projects/ts_forecasting/data/test.parquet")
df_test = set_cat(df_test, cat_features)


In [11]:
test_preds_series = pd.Series(index=df_test.index, dtype=float)

for h in HORIZONS:
    h_test    = df_test[df_test['horizon'] == h]
    model     = models[h]
    features  = features_used[h]

    preds     = model.predict(h_test[features])
    preds     = pd.Series(preds, index=h_test.index)

    zero_mask = h_test['code'].isin(ZERO_CODES)
    preds[zero_mask] = 0.0

    test_preds_series[h_test.index] = preds
    print(f"  Horizon {h:>2d} | Test rows: {len(h_test):,}  (zero-coded rows: {zero_mask.sum():,})")

assert test_preds_series.isna().sum() == 0, "Some test rows were not predicted!"
print(f"\n  Total test predictions: {len(test_preds_series):,}")


  Horizon  1 | Test rows: 379,617  (zero-coded rows: 69,005)
  Horizon  3 | Test rows: 376,558  (zero-coded rows: 68,523)
  Horizon 10 | Test rows: 362,057  (zero-coded rows: 65,675)
  Horizon 25 | Test rows: 328,875  (zero-coded rows: 59,033)

  Total test predictions: 1,447,107


## Build & Save Submission

In [12]:
prediction_df = pd.DataFrame({
    'id':         df_test['id'].values,
    'prediction': test_preds_series.values
})

prediction_df.info(max_cols=10)
prediction_df.head()


<class 'pandas.DataFrame'>
RangeIndex: 1447107 entries, 0 to 1447106
Data columns (total 2 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   id          1447107 non-null  str    
 1   prediction  1447107 non-null  float64
dtypes: float64(1), str(1)
memory usage: 73.8 MB


,id,prediction
0,W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647,-0.042258
1,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647,-0.150509
2,W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647,-0.371755
3,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647,-0.011484
4,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3648,-0.141316


In [13]:
prediction_df.to_csv(
    f"/home/lingod/kaggle_projects/ts_forecasting/submissions/submission_{VERSION}.csv",
    index=False
)
print(f"Submission saved: submission_{VERSION}.csv")


Submission saved: submission_v5c_full.csv


## Summary
| Horizon | v5c best_iter | +5% buffer | Full data rows |
|---------|--------------|------------|----------------|
| 1 | 1036 | 1088 | ~1,394,653 |
| 3 | 423 | 444 | ~1,385,816 |
| 10 | 1300 | 1365 | ~1,337,236 |
| 25 | 780 | 819 | ~1,219,709 |

| Metric | v5c (val split) | v5c_full (no split) | LB |
|--------|----------------|--------------------|-----------|
| Val Overall | (fill in) | N/A | |
| LB | | | |
